# WBCBench 2026 — Aşama 8: Final Protokol Kontrolü

Bu notebook eğitim yapmaz. Split/sınıf dağılımını, dosya-adı çakışmalarını ve final deney protokolünü kaydeder. `phase2_test` modele verilmez.


In [1]:
from pathlib import Path
import json
from collections import Counter
ROOT=Path.cwd(); DATA=ROOT/"wbc_project"/"data"/"processed"; OUT=ROOT/"outputs"; OUT.mkdir(exist_ok=True)
TRAIN=DATA/"phase2_train"; VAL=DATA/"phase2_eval"; TEST=DATA/"phase2_test"
for p in [TRAIN,VAL,TEST]: assert p.exists(),p
print("ROOT:",ROOT)


ROOT: C:\Users\sarah\Desktop\scı5


In [2]:
from torchvision.datasets import ImageFolder
tr=ImageFolder(TRAIN); va=ImageFolder(VAL); te=ImageFolder(TEST)
assert tr.classes==va.classes==te.classes and len(tr.classes)==13
classes=tr.classes
print("Classes:",classes)
print("Train:",len(tr),"Validation:",len(va),"Test:",len(te))


Classes: ['BA', 'BL', 'BNE', 'EO', 'LY', 'MMY', 'MO', 'MY', 'PC', 'PLY', 'PMY', 'SNE', 'VLY']
Train: 49794 Validation: 10700 Test: 32934


In [3]:
ct,cv,ce=Counter(tr.targets),Counter(va.targets),Counter(te.targets)
rows=[]
print(f"{'Class':28s} {'Train':>8} {'Val':>8} {'Test':>8}")
for i,name in enumerate(classes):
    r={"class":name,"train":ct[i],"validation":cv[i],"test":ce[i]}; rows.append(r)
    print(f"{name[:28]:28s} {ct[i]:8d} {cv[i]:8d} {ce[i]:8d}")


Class                           Train      Val     Test
BA                                840      120      606
BL                               3398      824     1770
BNE                               680      142      252
EO                               1596      338      984
LY                              13772     3142     8598
MMY                               670       94      340
MO                               4908      932     3066
MY                                810      150      408
PC                                112       64       24
PLY                                22        2        4
PMY                               152       68       80
SNE                             22270     4612    16356
VLY                               564      212      446


In [4]:
# Hızlı split kontrolü: relative sınıf/dosya adlarının kesişimi.
def relnames(ds,root):
    return {str(Path(p).relative_to(root)).lower() for p,_ in ds.samples}
rn_tr,rn_va,rn_te=relnames(tr,TRAIN),relnames(va,VAL),relnames(te,TEST)
a=rn_tr&rn_va; b=rn_tr&rn_te; c=rn_va&rn_te
print("Aynı relative ad Train-Val:",len(a))
print("Aynı relative ad Train-Test:",len(b))
print("Aynı relative ad Val-Test:",len(c))
print("Not: Bu hızlı kontroldür; final kalite kontrolde içerik-hash denetimi ayrıca yapılabilir.")


Aynı relative ad Train-Val: 0
Aynı relative ad Train-Test: 0
Aynı relative ad Val-Test: 0
Not: Bu hızlı kontroldür; final kalite kontrolde içerik-hash denetimi ayrıca yapılabilir.


In [5]:
protocol={
 "study":"WBCBench 2026","classes":13,
 "splits":{"train":"phase2_train","validation":"phase2_eval","test":"phase2_test"},
 "primary_metric":"Macro-F1",
 "secondary_metrics":["Balanced Accuracy","Accuracy","per-class F1"],
 "seeds":[42,1,7],"input_size":[224,224],
 "models":{
   "ResNet-50":{"pretraining":"ImageNet","adaptation":"full fine-tuning"},
   "ConvNeXt-Tiny":{"pretraining":"ImageNet","adaptation":"full fine-tuning"},
   "RETFound":{"pretraining":"CFP retinal checkpoint","primary_adaptation":"full fine-tuning",
               "ablations":["linear probe","last-block partial fine-tuning"]}
 },
 "model_selection":"Best validation Macro-F1 per seed.",
 "test_policy":"No training, tuning, early stopping, or model selection on test.",
 "reporting":"Mean ± SD across seeds; save test predictions for CI/statistics.",
 "planned":["confusion matrix","per-class analysis","severity analysis","paired statistical comparison"],
 "smoke_results_are_final":False
}
(OUT/"WBCBench_2026_FINAL_PROTOCOL.json").write_text(json.dumps(protocol,ensure_ascii=False,indent=2),encoding="utf-8")
manifest={"classes":classes,"counts":rows,"totals":{"train":len(tr),"validation":len(va),"test":len(te)},
          "relative_name_overlap":{"train_val":len(a),"train_test":len(b),"val_test":len(c)}}
(OUT/"WBCBench_2026_DATASET_MANIFEST.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
print("✓ Protokol ve dataset manifesti outputs/ klasörüne kaydedildi.")


✓ Protokol ve dataset manifesti outputs/ klasörüne kaydedildi.


In [6]:
print("="*68)
print("AŞAMA 8 — FINAL PROTOKOL KONTROL SONUCU")
print("="*68)
print("Classes:",len(classes))
print("Train:",len(tr),"Validation:",len(va),"Test:",len(te))
print("Relative-name overlap T-V / T-Test / V-Test:",len(a),len(b),len(c))
print("Primary metric: Macro-F1")
print("Seeds: 42, 1, 7")
print("TEST: eğitim/tuning/model selection için kullanılmayacak.")
print("✓ PROTOKOL KİLİTLENDİ")


AŞAMA 8 — FINAL PROTOKOL KONTROL SONUCU
Classes: 13
Train: 49794 Validation: 10700 Test: 32934
Relative-name overlap T-V / T-Test / V-Test: 0 0 0
Primary metric: Macro-F1
Seeds: 42, 1, 7
TEST: eğitim/tuning/model selection için kullanılmayacak.
✓ PROTOKOL KİLİTLENDİ
